In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.spatial.distance import cdist
from matbench.bench import MatbenchBenchmark
from sklearn.metrics import mean_absolute_error

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

np.math = math  # برای سازگاری احتمالی با matbench

# =============================================================================
# Optional / best-effort: space group
# =============================================================================
try:
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
except ImportError:
    SpacegroupAnalyzer = None

# =============================================================================
# Load Matbench dataset
# =============================================================================
mb = MatbenchBenchmark(autoload=False)
task = mb.matbench_phonons
task.load()

# =============================================================================
# Load periodic table
# =============================================================================
df_ptable = pd.read_csv('/kaggle/input/ptable/ptable.csv')

df_ptable.fillna(0, inplace=True)
df_ptable.drop(
    ['electronic_configuration', 'name', 'block', 'lattice_structure', 'is_radioactive'],
    axis=1,
    inplace=True,
    errors='ignore',
)

# تبدیل همه ستون‌ها (به‌جز symbol) به عدد
for col in df_ptable.columns:
    if col != 'symbol':
        df_ptable[col] = pd.to_numeric(df_ptable[col], errors='coerce')

df_ptable.fillna(0, inplace=True)

N_ATOM_FEATURES = df_ptable.shape[1] - 1  # بدون ستون symbol
N_BOND_FEATURES = 1                       # فقط: distance
EDGE_DIM = 1

print(f"✅ Loaded ptable: {N_ATOM_FEATURES} features/atom")


# =============================================================================
# Helpers
# =============================================================================
def _get_spacegroup_info(structure):
    """برگرداندن شماره گروه فضایی و سیستم کریستالی (به‌صورت عددی)"""
    spacegroup_num = 1
    lattice_type = 0  # cubic=0, tetragonal=1, ...

    if SpacegroupAnalyzer is None:
        return spacegroup_num, lattice_type

    try:
        sga = SpacegroupAnalyzer(structure)
        spacegroup_num = sga.get_space_group_number()
        crystal_system = sga.get_crystal_system()

        system_map = {
            'cubic': 0,
            'tetragonal': 1,
            'orthorhombic': 2,
            'hexagonal': 3,
            'trigonal': 4,
            'monoclinic': 5,
            'triclinic': 6,
        }
        lattice_type = system_map.get(crystal_system, 0)
    except Exception:
        pass

    return spacegroup_num, lattice_type


def _make_global_features(structure, num_sites):
    """
    ساخت ویژگی‌های سراسری (u) بر اساس حجم و تقارن:
      1) volume_per_atom
      2) spacegroup_num / 230.0
      3) lattice_type / 6.0
      4) lattice.volume
    """
    lattice = structure.lattice
    spacegroup_num, lattice_type = _get_spacegroup_info(structure)

    u = torch.tensor(
        [[
            structure.volume / num_sites,
            spacegroup_num / 230.0,
            lattice_type / 6.0,
            lattice.volume,
        ]],
        dtype=torch.float,
    )
    return u


def _compute_directed_neighbors_all_pairs(structure):
    """
    برای هر جفت (i, j) در یونیت‌سل، نزدیک‌ترین تصویر پریودیک j نسبت به i را پیدا می‌کند.
    خروجی:
      directed_dist[(i, j)] = d_ij (فاصله‌ی nearest image)
      neighbor_r[(i, j)]    = بردار از i به nearest image j (کارتزین)
      neighbor_list[i]      = مجموعه‌ای از jهایی که با i در ارتباط‌اند
    گراف نهایی کامل است (برای هر i<j یک bond).
    """
    n_sites = len(structure)
    lattice = structure.lattice
    frac_coords = np.array([site.frac_coords for site in structure.sites], dtype=np.float64)

    directed_dist = {}
    neighbor_r = {}
    neighbor_list = [set() for _ in range(n_sites)]

    for i in range(n_sites):
        fi = frac_coords[i]
        for j in range(n_sites):
            if i == j:
                continue

            fj = frac_coords[j]

            # d: کوتاه‌ترین فاصله پریودیک
            # image: بردار ترجمه (n1, n2, n3) که این فاصله را می‌سازد
            d, image = lattice.get_distance_and_image(fi, fj)

            # بردار از i به تصویر j در مختصات کسری
            vec_frac = fj + image - fi
            # تبدیل به کارتزین
            vec_cart = lattice.get_cartesian_coords(vec_frac).astype(np.float32)

            directed_dist[(i, j)] = float(d)
            neighbor_r[(i, j)] = vec_cart
            neighbor_list[i].add(j)

    return directed_dist, neighbor_r, neighbor_list


# =============================================================================
# Graph builders
# =============================================================================
def structure_to_graph(structure, target_value, df_ptable_unused=None, cutoff=50.0):
    """
    تبدیل ساختار به گراف پیوندی روی یونیت‌سل:
        nodes = bonds (i-j) برای هر جفت اتم در یونیت‌سل (i<j)
        edges = angles (i-k-j) با استفاده از بردارهای پریودیک نزدیک‌ترین تصویر
    شعاع کات‌آف عملاً استفاده نمی‌شود؛ همه‌ی جفت‌ها در نظر گرفته می‌شوند،
    ولی فاصله‌ها همگی nearest-image دقیق هستند.
    """
    structure = structure.copy()
    positions = np.array([site.coords for site in structure.sites], dtype=np.float32)
    n_sites = len(positions)

    # همسایه‌ها با PBC و نزدیک‌ترین تصویر برای همه‌ی جفت‌ها
    directed_dist, neighbor_r, neighbor_list = _compute_directed_neighbors_all_pairs(structure)

    # Bonds: یک پیوند برای هر جفت (i<j)
    bonds = []
    pair_dist = {}  # (i, j) با i<j -> d_ij (nearest image)

    for i in range(n_sites):
        for j in range(i + 1, n_sites):
            d_candidates = []
            if (i, j) in directed_dist:
                d_candidates.append(directed_dist[(i, j)])
            if (j, i) in directed_dist:
                d_candidates.append(directed_dist[(j, i)])

            if not d_candidates:
                continue

            d_ij = min(d_candidates)
            pair_dist[(i, j)] = d_ij
            bonds.append((i, j))

    y = torch.tensor([target_value], dtype=torch.float)

    # حالت بدون bond
    if len(bonds) == 0:
        x = torch.zeros((1, N_BOND_FEATURES), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, EDGE_DIM), dtype=torch.float)
        u = _make_global_features(structure, n_sites)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, u=u)

    # Node features (bonds) — فقط distance
    node_features = []
    for i, j in bonds:
        bond_distance = pair_dist[(i, j)]

        bond_feature = np.array(
            [
                bond_distance,  # تنها فیچر نود پیوندی
            ],
            dtype=np.float32,
        )
        node_features.append(bond_feature)

    x = torch.tensor(node_features, dtype=torch.float)

    # Edges between bonds (angles) با PBC
    edge_index = []
    edge_attr = []

    def get_vec(shared, other):
        if (shared, other) in neighbor_r:
            return neighbor_r[(shared, other)]
        if (other, shared) in neighbor_r:
            return -neighbor_r[(other, shared)]
        return positions[other] - positions[shared]

    for bond_idx1, (i1, j1) in enumerate(bonds):
        for bond_idx2, (i2, j2) in enumerate(bonds):
            if bond_idx1 == bond_idx2:
                continue

            shared_atom = None
            if i1 in (i2, j2):
                shared_atom = i1
                other1 = j1
                other2 = j2 if i1 == i2 else i2
            elif j1 in (i2, j2):
                shared_atom = j1
                other1 = i1
                other2 = j2 if j1 == i2 else i2

            if shared_atom is None:
                continue

            vec1 = get_vec(shared_atom, other1)
            vec2 = get_vec(shared_atom, other2)

            denom = np.linalg.norm(vec1) * np.linalg.norm(vec2)
            if denom == 0:
                continue

            cos_angle = np.dot(vec1, vec2) / denom
            cos_angle = np.clip(cos_angle, -1.0, 1.0)
            angle = np.arccos(cos_angle)

            edge_index.append([bond_idx1, bond_idx2])
            edge_attr.append([angle])

    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, EDGE_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    u = _make_global_features(structure, n_sites)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, u=u)


def structure_to_graph2(structure, target_value, df_ptable, cutoff=50.0):
    """
    تبدیل ساختار به گراف اتمی روی یونیت‌سل:
        Nodes = atoms (primitive cell)
        Edges = فاصله‌ی nearest-image بین هر جفت اتم (i ≠ j).
    گراف کامل است؛ برای هر i≠j یک یال داریم.
    """
    structure = structure.copy()

    # نودها = اتم‌های یونیت‌سل
    node_features = []
    atomic_numbers = []

    for site in structure.sites:
        Z = site.specie.Z
        atomic_numbers.append(Z)
        features = df_ptable.iloc[Z - 1, 1:].values.astype(np.float32)
        node_features.append(features)

    x = torch.tensor(node_features, dtype=torch.float)
    z = torch.tensor(atomic_numbers, dtype=torch.long)

    positions = np.array([site.coords for site in structure.sites], dtype=np.float32)
    pos = torch.tensor(positions, dtype=torch.float)
    n_atoms = len(positions)

    lattice = structure.lattice
    frac_coords = np.array([site.frac_coords for site in structure.sites], dtype=np.float64)

    edge_index = []
    edge_attr = []

    for i in range(n_atoms):
        fi = frac_coords[i]
        for j in range(n_atoms):
            if i == j:
                continue
            fj = frac_coords[j]

            d, image = lattice.get_distance_and_image(fi, fj)
            edge_index.append([i, j])
            edge_attr.append([float(d)])

    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, EDGE_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    y = torch.tensor([target_value], dtype=torch.float)
    u = torch.tensor([[structure.volume / n_atoms]], dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, pos=pos, z=z, u=u)


# =============================================================================
# GNN layers
# =============================================================================
class CustomMessagePassing(nn.Module):
    """
    Custom Message Passing with Attention:
    message = attention_weight * (neighbor_node_feature * edge_feature)
    """

    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2),
        )

        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        from torch_geometric.utils import softmax

        source_nodes = edge_index[0]
        target_nodes = edge_index[1]

        neighbor_features = x[source_nodes]  # [E, H]
        target_features = x[target_nodes]    # [E, H]

        attention_input = torch.cat(
            [target_features, neighbor_features, edge_attr], dim=1
        )  # [E, 3H]
        attention_scores = self.attention_mlp(attention_input)  # [E, 1]

        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr  # [E, H]
        weighted_messages = messages * attention_weights  # [E, H]

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        output = self.message_mlp(aggregated)
        return output


class DualGraphGNN(nn.Module):
    """
    Dual graph model: bond graph + atom graph
    با hidden جداگانه برای هر گراف
    """

    def __init__(
        self,
        n_bond_features=N_BOND_FEATURES,
        n_atom_features=N_ATOM_FEATURES,
        edge_dim=EDGE_DIM,
        bond_hidden=32,
        atom_hidden=64,
    ):
        super().__init__()

        self.bond_hidden = bond_hidden
        self.atom_hidden = atom_hidden

        # Bond branch ----------------------------------------------------------
        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, bond_hidden),
            nn.BatchNorm1d(bond_hidden),
            nn.SiLU(),
            nn.Dropout(0.1),
        )

        self.bond_edge_embedding = nn.Sequential(
            nn.Linear(edge_dim, bond_hidden),
            nn.SiLU(),
        )

        self.bond_message_layers = nn.ModuleList(
            [CustomMessagePassing(bond_hidden) for _ in range(5)]
        )
        self.bond_layer_norms = nn.ModuleList(
            [nn.LayerNorm(bond_hidden) for _ in range(5)]
        )

        self.bond_attention = nn.Sequential(
            nn.Linear(bond_hidden, bond_hidden // 4),
            nn.SiLU(),
            nn.Linear(bond_hidden // 4, 1),
            nn.Sigmoid(),
        )

        # Atom branch ----------------------------------------------------------
        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, atom_hidden),
            nn.BatchNorm1d(atom_hidden),
            nn.SiLU(),
            nn.Dropout(0.1),
        )

        self.atom_edge_embedding = nn.Sequential(
            nn.Linear(edge_dim, atom_hidden),
            nn.SiLU(),
        )

        self.atom_message_layers = nn.ModuleList(
            [CustomMessagePassing(atom_hidden) for _ in range(2)]
        )
        self.atom_layer_norms = nn.ModuleList(
            [nn.LayerNorm(atom_hidden) for _ in range(2)]
        )

        self.atom_attention = nn.Sequential(
            nn.Linear(atom_hidden, atom_hidden // 4),
            nn.SiLU(),
            nn.Linear(atom_hidden // 4, 1),
            nn.Sigmoid(),
        )

        # Learnable residual weights ------------------------------------------
        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        # Pooling --------------------------------------------------------------
        from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool

        self.bond_set2set_pool = Set2Set(bond_hidden, processing_steps=1)
        self.atom_set2set_pool = Set2Set(atom_hidden, processing_steps=1)

        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool

        # Global features ------------------------------------------------------
        # u: الان 4 فیچر دارد (در structure_to_graph)
        global_hidden = max(4, (bond_hidden + atom_hidden) // 4)
        self.global_hidden = global_hidden

        self.global_mlp = nn.Sequential(
            nn.Linear(4, global_hidden),
            nn.SiLU(),
        )

        # Final MLP ------------------------------------------------------------
        # اینجا از mean-bond + mean-atom + global استفاده شده
        final_input_dim = 1 * bond_hidden + 1 * atom_hidden + global_hidden

        self.final_mlp = nn.Sequential(
            nn.Linear(final_input_dim, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

    def forward(self, bond_data, atom_data):
        # Bond branch ----------------------------------------------------------
        x_bond = self.bond_embedding(bond_data.x)
        edge_features_bond = self.bond_edge_embedding(bond_data.edge_attr)

        for i, (message_layer, layer_norm) in enumerate(
            zip(self.bond_message_layers, self.bond_layer_norms)
        ):
            x_residual = x_bond
            x_bond = message_layer(x_bond, bond_data.edge_index, edge_features_bond)
            x_bond = layer_norm(x_bond)
            if i > 0:
                x_bond = x_bond + self.bond_residual_weight * x_residual
            x_bond = F.silu(x_bond)

        attention_weights_bond = self.bond_attention(x_bond)
        x_bond_weighted = x_bond * attention_weights_bond

        bond_set2set = self.bond_set2set_pool(x_bond_weighted, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)

        # Atom branch ----------------------------------------------------------
        x_atom = self.atom_embedding(atom_data.x)
        edge_features_atom = self.atom_edge_embedding(atom_data.edge_attr)

        for i, (message_layer, layer_norm) in enumerate(
            zip(self.atom_message_layers, self.atom_layer_norms)
        ):
            x_residual = x_atom
            x_atom = message_layer(x_atom, atom_data.edge_index, edge_features_atom)
            x_atom = layer_norm(x_atom)
            if i > 0:
                x_atom = x_atom + self.atom_residual_weight * x_residual
            x_atom = F.silu(x_atom)

        attention_weights_atom = self.atom_attention(x_atom)
        x_atom_weighted = x_atom * attention_weights_atom

        atom_set2set = self.atom_set2set_pool(x_atom_weighted, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)

        # Global ---------------------------------------------------------------
        global_features = self.global_mlp(bond_data.u)

        # Combine --------------------------------------------------------------
        combined = torch.cat(
            [
                #bond_set2set,
                bond_mean,
                #bond_max,
                #atom_set2set,
                atom_mean,
                #atom_max,
                global_features,
            ],
            dim=1,
        )

        out = self.final_mlp(combined)
        return out.squeeze()


# =============================================================================
# Training utilities
# =============================================================================
def evaluate(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for bond_data, atom_data in loader:
            bond_data = bond_data.to(device)
            atom_data = atom_data.to(device)
            pred = model(bond_data, atom_data).cpu().numpy()
            target = bond_data.y.cpu().numpy()
            preds.extend(pred)
            targets.extend(target)
    return mean_absolute_error(targets, preds)


print("✅ Training functions ready")

# =============================================================================
# Cross-fold training (no test-based checkpointing)
# =============================================================================
from tqdm import tqdm

NUM_EPOCHS = 2500
LOG_EVERY = 10  # هر ۱۰ ایپاک لاگ متریک

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

fold_maes = []

# تعداد فولدهای تسک، ولی حداکثر تا فولد ۵ (۰ تا ۵)
num_all_folds = len(task.folds) if hasattr(task, "folds") else 5
max_folds = min(6, num_all_folds)

for fold_num in range(max_folds):
    print(f"\n================ Fold {fold_num} / {max_folds-1} ================\n")

    # =============================================================================
    # Data preparation for this fold
    # =============================================================================
    train_inputs, train_outputs = task.get_train_and_val_data(fold_num)
    test_inputs, test_outputs = task.get_test_data(fold_num, include_target=True)

    print(f"Converting {len(train_inputs)} train structures to graphs...")
    train_bond_graphs, train_atom_graphs = [], []
    for structure, target in tqdm(
        zip(train_inputs, train_outputs), total=len(train_inputs)
    ):
        train_bond_graphs.append(
            structure_to_graph(structure, target, df_ptable)
        )
        train_atom_graphs.append(
            structure_to_graph2(structure, target, df_ptable)
        )

    print(f"Converting {len(test_inputs)} test structures to graphs...")
    test_bond_graphs, test_atom_graphs = [], []
    for structure, target in tqdm(
        zip(test_inputs, test_outputs), total=len(test_inputs)
    ):
        test_bond_graphs.append(
            structure_to_graph(structure, target, df_ptable)
        )
        test_atom_graphs.append(
            structure_to_graph2(structure, target, df_ptable)
        )

    train_dataset = list(zip(train_bond_graphs, train_atom_graphs))
    test_dataset = list(zip(test_bond_graphs, test_atom_graphs))

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # =============================================================================
    # Model, optimizer, scheduler
    # =============================================================================
    model = DualGraphGNN(
        n_bond_features=N_BOND_FEATURES,
        n_atom_features=N_ATOM_FEATURES,
        edge_dim=EDGE_DIM,
        bond_hidden=64,   # hidden گراف پیوندی
        atom_hidden=32,   # hidden گراف اتمی
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"✅ Dual Graph GNN Model (fold {fold_num}): {n_params:,} params")
    print("   Features:")
    print("   - Bond graph (nodes=bonds, edges=angles)")
    print("   - Atom graph (nodes=atoms, edges=distances)")
    print("   - Separate message passing for each graph with different hidden sizes")
    print("   - Combined pooling + MLP")

    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.01,
        epochs=NUM_EPOCHS,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        anneal_strategy='cos',
        div_factor=25.0,
        final_div_factor=1e5,
    )

    # =============================================================================
    # Training loop (full NUM_EPOCHS، بدون استفاده از تست برای ذخیره/قطع زودهنگام)
    # =============================================================================
    print("\n🚀 Training dual graph model...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0.0

        for bond_data, atom_data in train_loader:
            bond_data = bond_data.to(device)
            atom_data = atom_data.to(device)

            optimizer.zero_grad()
            pred = model(bond_data, atom_data)
            loss = F.mse_loss(pred, bond_data.y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item() * bond_data.num_graphs

        train_loss = total_loss / len(train_dataset)

        # هر ۱۰ ایپاک: لاگ متریک روی ترین و تست (فقط گزارش، بدون استفاده در آموزش)
        if epoch % LOG_EVERY == 0:
            train_mae = evaluate(model, train_loader, device)
            test_mae = evaluate(model, test_loader, device)
            current_lr = optimizer.param_groups[0]['lr']
            print(
                f"Fold {fold_num} | Epoch {epoch:4d} | "
                f"Train loss: {train_loss:.4f} | "
                f"Train MAE: {train_mae:.4f} | "
                f"Test MAE: {test_mae:.4f} | "
                f"LR: {current_lr:.6f}"
            )

    # =============================================================================
    # Evaluation روی تست فقط بعد از آخرین ایپاک (برای گزارش نهایی فولد)
    # =============================================================================
    test_mae = evaluate(model, test_loader, device)
    fold_maes.append(test_mae)
    print(f"\n✅ Fold {fold_num} MAE (final): {test_mae:.4f} cm⁻¹")

# =============================================================================
# Summary over folds
# =============================================================================
mean_mae = sum(fold_maes) / len(fold_maes)
print("\n================ Cross-fold results ================")
for i, m in enumerate(fold_maes):
    print(f"Fold {i}: MAE = {m:.4f} cm⁻¹")
print(f"\n📊 Mean MAE over {len(fold_maes)} folds: {mean_mae:.4f} cm⁻¹")


# Physics informed preatrain to learn 2 body and 3 body interactions without using any angles in graphs

In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from matbench.bench import MatbenchBenchmark
from sklearn.metrics import mean_absolute_error

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

np.math = math  # برای سازگاری احتمالی با matbench

# =============================================================================
# Optional / best-effort: space group
# =============================================================================
try:
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
except ImportError:
    SpacegroupAnalyzer = None

# =============================================================================
# Load Matbench dataset
# =============================================================================
mb = MatbenchBenchmark(autoload=False)
task = mb.matbench_phonons
task.load()

# =============================================================================
# Load periodic table
# =============================================================================
df_ptable = pd.read_csv('/kaggle/input/ptable-pruned/ptable_pruned.csv')

df_ptable.fillna(0, inplace=True)
df_ptable.drop(
    ['electronic_configuration', 'name', 'block', 'lattice_structure', 'is_radioactive'],
    axis=1,
    inplace=True,
    errors='ignore',
)

for col in df_ptable.columns:
    if col != 'symbol':
        df_ptable[col] = pd.to_numeric(df_ptable[col], errors='coerce')

df_ptable.fillna(0, inplace=True)

N_ATOM_FEATURES = df_ptable.shape[1] - 1
EDGE_DIM = 1

print(f"Loaded ptable: {N_ATOM_FEATURES} features/atom")

# =============================================================================
# Synthetic 2-body / 3-body potential parameters
# =============================================================================
rng = np.random.default_rng(0)
MAX_Z = 100

a_Z = rng.normal(size=MAX_Z + 1).astype(np.float32)  # برای E2
b_Z = rng.normal(size=MAX_Z + 1).astype(np.float32)  # برای E3

mu2 = 2.0
sigma2 = 0.8
mu3 = 6.0
sigma3 = 1.5


def compute_synthetic_E2_E3(atomic_numbers, pair_indices, pair_dist):
    """
    atomic_numbers: list/array of Z, len=N
    pair_indices: list of (i, j) with i < j
    pair_dist: list of d_ij
    """
    Z = np.asarray(atomic_numbers, dtype=int)
    pair_indices = np.asarray(pair_indices, dtype=int)
    pair_dist = np.asarray(pair_dist, dtype=float)

    # E2
    E2 = 0.0
    for (i, j), rij in zip(pair_indices, pair_dist):
        Zi = Z[i]
        Zj = Z[j]
        if Zi > MAX_Z or Zj > MAX_Z:
            continue
        w = a_Z[Zi] * a_Z[Zj]
        E2 += float(w * np.exp(-((rij - mu2) ** 2) / (sigma2 ** 2)))

    # E3
    dist_map = {}
    for (i, j), rij in zip(pair_indices, pair_dist):
        dist_map[(i, j)] = rij

    n_atoms = len(Z)
    E3 = 0.0
    for i in range(n_atoms):
        for j in range(i + 1, n_atoms):
            for k in range(j + 1, n_atoms):
                rij = dist_map.get((i, j), None)
                rik = dist_map.get((i, k), None)
                rjk = dist_map.get((j, k), None)
                if rij is None or rik is None or rjk is None:
                    continue
                s = rij + rik + rjk
                Zi, Zj, Zk = Z[i], Z[j], Z[k]
                if Zi > MAX_Z or Zj > MAX_Z or Zk > MAX_Z:
                    continue
                w3 = b_Z[Zi] * b_Z[Zj] * b_Z[Zk]
                E3 += float(w3 * np.exp(-((s - mu3) ** 2) / (sigma3 ** 2)))

    return E2, E3


# =============================================================================
# Helpers
# =============================================================================
def _get_spacegroup_info(structure):
    spacegroup_num = 1
    lattice_type = 0

    if SpacegroupAnalyzer is None:
        return spacegroup_num, lattice_type

    try:
        sga = SpacegroupAnalyzer(structure)
        spacegroup_num = sga.get_space_group_number()
        crystal_system = sga.get_crystal_system()

        system_map = {
            'cubic': 0,
            'tetragonal': 1,
            'orthorhombic': 2,
            'hexagonal': 3,
            'trigonal': 4,
            'monoclinic': 5,
            'triclinic': 6,
        }
        lattice_type = system_map.get(crystal_system, 0)
    except Exception:
        pass

    return spacegroup_num, lattice_type


def _make_global_features(structure, num_sites):
    lattice = structure.lattice
    spacegroup_num, lattice_type = _get_spacegroup_info(structure)

    u = torch.tensor(
        [[
            structure.volume / num_sites,
            spacegroup_num / 230.0,
            lattice_type / 6.0,
            lattice.volume,
        ]],
        dtype=torch.float,
    )
    return u


# =============================================================================
# Graph builder with synthetic E2/E3
# =============================================================================
def structure_to_graph2(structure, target_value, df_ptable, cutoff=50.0):
    structure = structure.copy()

    node_features = []
    atomic_numbers = []

    for site in structure.sites:
        Z = site.specie.Z
        atomic_numbers.append(Z)
        features = df_ptable.iloc[Z - 1, 1:].values.astype(np.float32)
        node_features.append(features)

    x = torch.tensor(node_features, dtype=torch.float)
    z = torch.tensor(atomic_numbers, dtype=torch.long)

    positions = np.array([site.coords for site in structure.sites], dtype=np.float32)
    pos = torch.tensor(positions, dtype=torch.float)
    n_atoms = len(positions)

    lattice = structure.lattice
    frac_coords = np.array([site.frac_coords for site in structure.sites], dtype=np.float64)

    # Directed edges برای GNN
    edge_index = []
    edge_attr = []

    for i in range(n_atoms):
        fi = frac_coords[i]
        for j in range(n_atoms):
            if i == j:
                continue
            fj = frac_coords[j]
            d, image = lattice.get_distance_and_image(fi, fj)
            edge_index.append([i, j])
            edge_attr.append([float(d)])

    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, EDGE_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    # Pair list برای E2/E3 (i < j)
    pair_indices = []
    pair_dist = []
    for i in range(n_atoms):
        fi = frac_coords[i]
        for j in range(i + 1, n_atoms):
            fj = frac_coords[j]
            d, image = lattice.get_distance_and_image(fi, fj)
            pair_indices.append((i, j))
            pair_dist.append(float(d))

    E2_synth, E3_synth = compute_synthetic_E2_E3(atomic_numbers, pair_indices, pair_dist)

    y_real = torch.tensor([target_value], dtype=torch.float)
    y2_synth = torch.tensor([E2_synth], dtype=torch.float)
    y3_synth = torch.tensor([E3_synth], dtype=torch.float)

    u = _make_global_features(structure, n_atoms)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y_real,
        pos=pos,
        z=z,
        u=u,
    )
    data.y2_synth = y2_synth
    data.y3_synth = y3_synth

    return data


# =============================================================================
# GNN layers
# =============================================================================
class CustomMessagePassing(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2),
        )

        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        from torch_geometric.utils import softmax

        source_nodes = edge_index[0]
        target_nodes = edge_index[1]

        neighbor_features = x[source_nodes]  # [E, H]
        target_features = x[target_nodes]    # [E, H]

        attention_input = torch.cat(
            [target_features, neighbor_features, edge_attr], dim=1
        )  # [E, 3H]
        attention_scores = self.attention_mlp(attention_input)  # [E, 1]

        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr  # [E, H]
        weighted_messages = messages * attention_weights  # [E, H]

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        output = self.message_mlp(aggregated)
        return output


class AtomGraphGNN(nn.Module):
    """
    - بک‌بون: امبدینگ + دو لایه مسیج‌پسینگ
    - h_final: concat(mean_l1, max_l1, mean_l2, max_l2, global)
    - head سینتتیک (تغییر داده شده):
        E2 از pooling لایه 1
        E3 از pooling لایه 2
    - head اصلی matbench:
        y_pred = MLP(h_final)
    """

    def __init__(
        self,
        n_atom_features=N_ATOM_FEATURES,
        edge_dim=EDGE_DIM,
        atom_hidden=64,
        bond_hidden=32,
    ):
        super().__init__()

        self.atom_hidden = atom_hidden

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, atom_hidden),
            nn.BatchNorm1d(atom_hidden),
            nn.SiLU(),
            nn.Dropout(0.1),
        )

        self.atom_edge_embedding = nn.Sequential(
            nn.Linear(edge_dim, atom_hidden),
            nn.SiLU(),
        )

        self.atom_message_layers = nn.ModuleList(
            [CustomMessagePassing(atom_hidden) for _ in range(2)]
        )
        self.atom_layer_norms = nn.ModuleList(
            [nn.LayerNorm(atom_hidden) for _ in range(2)]
        )

        global_hidden = max(4, atom_hidden // 2)
        self.global_hidden = global_hidden

        self.global_mlp = nn.Sequential(
            nn.Linear(4, global_hidden),
            nn.SiLU(),
        )

        from torch_geometric.nn import global_mean_pool, global_max_pool
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool

        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        # h_final = [mean_l1, max_l1, mean_l2, max_l2, global]
        self.final_input_dim = 4 * atom_hidden + global_hidden

        # -------------------- تغییرات فقط در مدل --------------------
        # head سینتتیک: E2 فقط از pooling لایه1، E3 فقط از pooling لایه2
        aux_in_dim = 2 * atom_hidden + global_hidden
        self.e2_head = nn.Linear(aux_in_dim, 1)
        self.e3_head = nn.Linear(aux_in_dim, 1)
        # -----------------------------------------------------------

        # head اصلی matbench: h_final -> scalar
        self.matbench_head = nn.Sequential(
            nn.Linear(self.final_input_dim, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

    def forward(self, atom_data):
        x_atom = self.atom_embedding(atom_data.x)
        edge_features_atom = self.atom_edge_embedding(atom_data.edge_attr)

        x_l1 = None
        x_l2 = None

        for i, (message_layer, layer_norm) in enumerate(
            zip(self.atom_message_layers, self.atom_layer_norms)
        ):
            x_residual = x_atom
            x_atom = message_layer(x_atom, atom_data.edge_index, edge_features_atom)
            x_atom = layer_norm(x_atom)
            if i > 0:
                x_atom = x_atom + self.atom_residual_weight * x_residual
            x_atom = F.silu(x_atom)

            if i == 0:
                x_l1 = x_atom
            if i == 1:
                x_l2 = x_atom

        if x_l1 is None:
            x_l1 = x_atom
        if x_l2 is None:
            x_l2 = x_atom

        global_features = self.global_mlp(atom_data.u)

        mean_l1 = self.mean_pool(x_l1, atom_data.batch)
        max_l1 = self.max_pool(x_l1, atom_data.batch)
        mean_l2 = self.mean_pool(x_l2, atom_data.batch)
        max_l2 = self.max_pool(x_l2, atom_data.batch)

        # -------------------- تغییرات فقط در مدل --------------------
        # E2 از لایه1، E3 از لایه2
        h1 = torch.cat([mean_l1, max_l1, global_features], dim=1)
        h2 = torch.cat([mean_l2, max_l2, global_features], dim=1)

        E2_pred = self.e2_head(h1).squeeze(-1)
        E3_pred = self.e3_head(h2).squeeze(-1)
        # -----------------------------------------------------------

        h_final = torch.cat(
            [mean_l1, max_l1, mean_l2, max_l2, global_features],
            dim=1,
        )  # [B, final_input_dim]

        # head اصلی matbench
        y_pred = self.matbench_head(h_final).squeeze(-1)  # [B]

        return y_pred, E2_pred, E3_pred


# =============================================================================
# Training utilities
# =============================================================================
def evaluate_real(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            y_pred, E2_pred, E3_pred = model(data)
            pred = y_pred.detach().cpu().numpy()
            target = data.y.detach().cpu().numpy()
            preds.extend(pred)
            targets.extend(target)
    return mean_absolute_error(targets, preds)


print("Model and utils ready")

# =============================================================================
# K-fold training over 5 folds
# =============================================================================
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
fold_maes = []

pre_epochs = 200
finetune_epochs = 1500
lambda_aux = 0.5

for fold_num in range(5):
    print("\n" + "=" * 80)
    print(f"Starting fold {fold_num}")
    print("=" * 80)

    # --------------------- Data preparation for this fold --------------------
    train_inputs, train_outputs = task.get_train_and_val_data(fold_num)
    test_inputs, test_outputs = task.get_test_data(fold_num, include_target=True)

    print(f"Converting {len(train_inputs)} train structures to ATOM graphs (fold {fold_num})")
    train_graphs = []
    for structure, target in tqdm(
        zip(train_inputs, train_outputs), total=len(train_inputs)
    ):
        train_graphs.append(
            structure_to_graph2(structure, target, df_ptable)
        )

    print(f"Converting {len(test_inputs)} test structures to ATOM graphs (fold {fold_num})")
    test_graphs = []
    for structure, target in tqdm(
        zip(test_inputs, test_outputs), total=len(test_inputs)
    ):
        test_graphs.append(
            structure_to_graph2(structure, target, df_ptable)
        )

    train_dataset = train_graphs
    test_dataset = test_graphs

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # --------------------------- Model init ----------------------------------
    model = AtomGraphGNN(
        n_atom_features=N_ATOM_FEATURES,
        edge_dim=EDGE_DIM,
        atom_hidden=64,
        bond_hidden=32,
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"[Fold {fold_num}] Model params: {n_params:,}")

    # --------------------------- Phase 1: Pretrain ---------------------------
    print(f"[Fold {fold_num}] Pretraining on synthetic 2-body and 3-body energies")

    optimizer_pre = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler_pre = torch.optim.lr_scheduler.OneCycleLR(
        optimizer_pre,
        max_lr=0.003,
        epochs=pre_epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.2,
        anneal_strategy='cos',
        div_factor=10.0,
        final_div_factor=1e3,
    )

    for epoch in range(pre_epochs):
        model.train()
        total_loss = 0.0
        for data in train_loader:
            data = data.to(device)
            optimizer_pre.zero_grad()

            y_pred, E2_pred, E3_pred = model(data)
            y2 = data.y2_synth.to(device).view(-1)
            y3 = data.y3_synth.to(device).view(-1)

            loss_pre = F.mse_loss(E2_pred, y2) + F.mse_loss(E3_pred, y3)

            loss_pre.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer_pre.step()
            scheduler_pre.step()

            total_loss += loss_pre.item() * data.num_graphs

        avg_loss = total_loss / len(train_dataset)
        if epoch % 20 == 0 or epoch == pre_epochs - 1:
            print(f"[Fold {fold_num}][Pretrain] Epoch {epoch:3d} | Loss: {avg_loss:.4f}")

    print(f"[Fold {fold_num}] Pretraining done")

    # --------------------------- Phase 2: Finetune ---------------------------
    print(f"[Fold {fold_num}] Finetuning on matbench_phonons with main+aux loss")

    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.01,
        epochs=finetune_epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.2,
        anneal_strategy='cos',
        div_factor=25.0,
        final_div_factor=1e5,
    )

    for epoch in range(finetune_epochs):
        model.train()
        total_loss = 0.0

        for data in train_loader:
            data = data.to(device)

            optimizer.zero_grad()
            y_pred, E2_pred, E3_pred = model(data)

            y_real = data.y.to(device).view(-1)
            y2 = data.y2_synth.to(device).view(-1)
            y3 = data.y3_synth.to(device).view(-1)

            loss_main = F.mse_loss(y_pred, y_real)
            loss_aux = F.mse_loss(E2_pred, y2) + F.mse_loss(E3_pred, y3)
            loss = loss_main + lambda_aux * loss_aux

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item() * data.num_graphs

        train_loss = total_loss / len(train_dataset)

        # صرفاً به عنوان متریک لاگ
        if epoch % 5 == 0:
            test_mae = evaluate_real(model, test_loader, device)
            current_lr = optimizer.param_groups[0]['lr']
            print(
                f"[Fold {fold_num}] Epoch {epoch:4d} | "
                f"TrainLoss: {train_loss:.4f} | Test MAE: {test_mae:.4f} | LR: {current_lr:.6f}"
            )

    # دقت نهایی این فولد با آخرین ایپاک
    final_test_mae = evaluate_real(model, test_loader, device)
    fold_maes.append(final_test_mae)
    print(f"[Fold {fold_num}] Final Test MAE (last epoch): {final_test_mae:.4f} cm^-1")

# =============================================================================
# Overall 5-fold mean
# =============================================================================
fold_maes = np.array(fold_maes, dtype=float)
print("\n" + "=" * 80)
for i, mae in enumerate(fold_maes):
    print(f"Fold {i} final test MAE: {mae:.4f} cm^-1")
print(f"Mean 5-fold Test MAE (last epoch): {fold_maes.mean():.4f} cm^-1")
print("=" * 80)
